# SRF — LEAO

Terminal Colab atualizado.

**Setup (uma vez):**
1. Clique no icone de chave na barra lateral
2. Adicione `ORCA_PASSWORD` com a senha
3. Execute as 3 celulas em ordem


In [ ]:
#@title 1. Instalar SRF
import os, subprocess

PROJECT_DIR = '/content/srf'
REPO_URL = 'https://github.com/xAngryBadger/pip-forest.git'
BRANCH = 'refactor/v8'

if os.path.exists(PROJECT_DIR):
    print(f'[...] Removendo versao anterior...')
    subprocess.run(['rm', '-rf', PROJECT_DIR], capture_output=True)

print(f'[...] Clonando {REPO_URL} branch {BRANCH} ...')
r = subprocess.run(
    ['git', 'clone', '-b', BRANCH, REPO_URL, PROJECT_DIR],
    capture_output=True, text=True
)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError(f'git clone falhou (codigo {r.returncode})')
print('[OK] Clone concluido')

print('[...] Instalando dependencias ...')
r1 = subprocess.run(
    ['pip', 'install', '-q', '-r', os.path.join(PROJECT_DIR, 'requirements-web.txt')],
    capture_output=True, text=True
)
r2 = subprocess.run(
    ['pip', 'install', '-q', 'pandas', 'openpyxl', 'rich', 'colorama'],
    capture_output=True, text=True
)
if r1.returncode != 0 or r2.returncode != 0:
    print(r1.stderr)
    print(r2.stderr)
    raise RuntimeError('pip install falhou')
print('[OK] Dependencias instaladas')

In [ ]:
#@title 2. Iniciar servidor + tunnel
import subprocess, os, time, re, sys

PORT = 8000
TUNNEL_URL_FILE = '/tmp/srf_tunnel_url.txt'
UVICORN_LOG = '/tmp/srf_uvicorn.log'

# Matar processos anteriores
for p in ['uvicorn', 'cloudflared', 'atm_v6_3']:
    subprocess.run(['pkill', '-f', p], capture_output=True)
time.sleep(1)

# Iniciar uvicorn em background
uv = open(UVICORN_LOG, 'w')
subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'src.web.api:app',
     '--host', '0.0.0.0', '--port', str(PORT)],
    stdout=uv, stderr=uv, cwd='/content/cli_planilhas'
)
time.sleep(3)

# Tentar Colab port forwarding (nativo, sem DNS externo)
tunnel_url = None
try:
    from google.colab.output import eval_js
    # Colab proxy — URL direta do Google, bypassa DNS
    proxy_url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
    if proxy_url:
        tunnel_url = proxy_url.rstrip('/')
        # Set base path for Colab proxy
        os.environ['ORCA_BASE_PATH'] = '/proxy/' + str(PORT)
        print(f'Colab proxy: {tunnel_url}')
except Exception as e:
    print(f'Colab proxy falhou: {e}')

# Fallback: cloudflared (so funciona se DNS nao estiver bloqueado)
if not tunnel_url:
    print('Tentando cloudflared...')
    CF_LOG = '/tmp/srf_cloudflared.log'
    subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}',
         '--http2-origin'],
        stdout=open(CF_LOG, 'w'), stderr=subprocess.STDOUT
    )
    time.sleep(8)
    try:
        cf_log = open(CF_LOG).read()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', cf_log)
        if m:
            tunnel_url = m.group(0)
    except:
        pass

# Segundo fallback: ngrok
if not tunnel_url:
    print('Tentando ngrok...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], capture_output=True)
    ngrok_token = None
    try:
        from google.colab import userdata
        ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    except:
        pass
    if ngrok_token:
        from pyngrok import ngrok as ngrok_mod
        ngrok_mod.set_auth_token(ngrok_token)
        tunnel = ngrok_mod.connect(PORT, 'http')
        tunnel_url = tunnel.public_url
    else:
        print('NGROK_AUTHTOKEN nao encontrado nos segredos.')
        print('Crie conta gratis: https://dashboard.ngrok.com/signup')
        print('Token: https://dashboard.ngrok.com/get-started/your-authtoken')

if tunnel_url:
    with open(TUNNEL_URL_FILE, 'w') as f:
        f.write(tunnel_url)
    print(f'\n{"=" * 60}')
    print(f'  URL: {tunnel_url}')
    print(f'  Login: {tunnel_url}/login')
    print(f'{"=" * 60}')
else:
    print('ERRO: Nenhum tunnel disponivel!')
    print('Adicione NGROK_AUTHTOKEN nos segredos do Colab.')

# Teste rapido
import requests
try:
    r = requests.get(f'http://localhost:{PORT}/login', timeout=5)
    print(f'Localhost: {r.status_code} OK')
except Exception as e:
    print(f'Localhood falhou: {e}')


In [ ]:
#@title 3. Terminal SRF
import os
import IPython.display as disp

TUNNEL_URL_FILE = '/tmp/srf_tunnel_url.txt'

if not os.path.exists(TUNNEL_URL_FILE):
    raise RuntimeError('Execute a celula 2 primeiro para iniciar o servidor')

tunnel_url = open(TUNNEL_URL_FILE).read().strip()

print(f'URL: {tunnel_url}')
print('Digite a senha no iframe abaixo para acessar o terminal.')
print(f'Ou abra em nova aba: {tunnel_url}/app')

disp.HTML(f"""
<div style="border:3px solid #2D6A4F;box-shadow:4px 4px 0px #000;border-radius:4px;overflow:hidden;">
<div style="background:#2D6A4F;color:#fff;padding:6px 12px;font-family:monospace;font-size:13px;font-weight:bold;">
SRF Terminal &mdash; <a href="{tunnel_url}/app" style="color:#90EE90;text-decoration:none;" target="_blank">abrir em nova aba</a>
</div>
<iframe src="{tunnel_url}/app" style="width:100%;height:600px;border:0;"></iframe>
</div>
""")


In [ ]:
#@title 4. Diagnostico + Tunnel Alternativo
import os, subprocess, time, re

TUNNEL_URL_FILE = '/tmp/srf_tunnel_url.txt'
UVICORN_LOG = '/tmp/srf_uvicorn.log'
CF_LOG = '/tmp/srf_cloudflared.log'
PORT = 8000

print('=== TESTE DE DNS ===')
domains = ['trycloudflare.com', 'ngrok-free.app', 'loca.lt', 'bore.pub', 'serveo.net']
for d in domains:
    r = subprocess.run(['getent', 'hosts', d], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        ip = r.stdout.strip().split()[0]
        print(f'  {d}: OK ({ip})')
    else:
        print(f'  {d}: FALHOU')

if os.path.exists(TUNNEL_URL_FILE):
    tunnel_url = open(TUNNEL_URL_FILE).read().strip()
    print(f'\nTunnel URL: {tunnel_url}')
    import requests
    try:
        r = requests.get(f'{tunnel_url}/login', timeout=15)
        print(f'Tunnel HTTP: {r.status_code} (FUNCIONA!)')
    except Exception as e:
        print(f'Tunnel HTTP: FALHOU - {str(e)[:120]}')
        print('\n>>> Cloudflare tunnel bloqueado. Tentando ngrok...')
        subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
        time.sleep(1)
        print('[ngrok] Instalando...')
        subprocess.run(['pip', 'install', '-q', 'pyngrok'], capture_output=True)
        ngrok_token = None
        try:
            from google.colab import userdata
            ngrok_token = userdata.get('NGROK_AUTHTOKEN')
        except:
            pass
        if ngrok_token:
            from pyngrok import ngrok as ngrok_mod
            ngrok_mod.set_auth_token(ngrok_token)
            print('[ngrok] Conectando...')
            tunnel = ngrok_mod.connect(PORT, 'http')
            ngrok_url = tunnel.public_url
            with open(TUNNEL_URL_FILE, 'w') as f:
                f.write(ngrok_url)
            print(f'\n{"=" * 50}')
            print(f'  NGROK URL: {ngrok_url}')
            print(f'{"=" * 50}')
        else:
            print('>>> Sem NGROK_AUTHTOKEN. Crie uma conta gratis em:')
            print('>>> https://dashboard.ngrok.com/signup')
            print('>>> Pegue seu token em:')
            print('>>> https://dashboard.ngrok.com/get-started/your-authtoken')
            print('>>> Adicione como segredo NGROK_AUTHTOKEN no Colab')
else:
    print('Nenhum tunnel configurado. Execute a celula 2 primeiro.')

print('\n=== Uvicorn log (ultimas 10 linhas) ===')
if os.path.exists(UVICORN_LOG):
    for l in open(UVICORN_LOG).readlines()[-10:]:
        print(l, end='')
